#### a. Derive the corresponding update rule for the image matrix data:

Tenemos la ecuación de Poisson: $\Delta\phi(p) = f(p)$, donde $\phi\rightarrow data$ y $f\rightarrow rho$

Entonces queremos resolver: $\Delta data[i,j] = rho[i,j]$ 

El Laplaciano en 2D es: $\Delta=\dfrac{∂²}{∂x²} + \dfrac{∂²}{∂y²}$

Como trabajamos en una matriz, no tenemos derivadas continuas. Entonces aproximamos las segundas derivadas utilizando diferencias finitas.

Para la parte vertical (i): 

$\dfrac{∂²\phi}{∂x²} \approx data[i+1,j] - 2 data[i,j] + data[i-1,j]$

Para la horizontal (j): 

$\dfrac{∂²\phi}{∂y²} \approx data[i,j+1] - 2 data[i,j] + data[i,j-1]$

Entonces el Laplaciano nos queda: 

$\Delta data[i,j] \approx data[i+1,j] + data[i-1,j] + data[i,j+1] + data[i,j-1] - 4 data[i,j]$

Ahora usamos la ec. de Poisson y reemplazamos el Laplaciano: 

$\Delta data[i,j] = rho[i,j] = data[i+1,j] + data[i-1,j] + data[i,j+1] + data[i,j-1] - 4 data[i,j]$

Si despejamos $data[i,j]$ obtenemos: 

$data[i,j] =\dfrac{1}{4}(data[i+1,j] + data[i-1,j] + data[i,j+1] + data[i,j-1] - rho[i,j])$

#### b. Design an MPI algorithm that distributes the matrix data on a two-dimensional grid of tiles. Make use of derived datatypes in order to efficiently communicate columns between neighboring blocks.

La idea es dividir la matriz global $N \times N$ entre los procesos MPI usando una partición bidimensional de bloques, en lugar de asignar filas completas a cada proceso como en la versión 1D.

Para organizar los procesos en una grilla 2D se utiliza un comunicador cartesiano:

```c
int dims[2] = {0, 0};
MPI_Dims_create(size, 2, dims);
MPI_Cart_create(MPI_COMM_WORLD, 2, dims, periods, reorder, &cart_comm);
```

`MPI_Dims_create` elige automáticamente una grilla lo más cuadrada posible.

El comunicador cartesiano permite identificar a cada proceso por sus coordenadas $(c_x, c_y)$ dentro de la grilla, y obtener sus vecinos de forma natural con `MPI_Cart_shift`, sin calcular rangos manualmente.

Cada proceso es responsable de un bloque local de tamaño:

$$\text{local\_nx} = \frac{N}{d_x} \qquad \text{local\_ny} = \frac{N}{d_y}$$

donde $d_x \times d_y$ es el tamaño de la grilla de procesos.

Para poder aplicar la regla de Jacobi en los bordes del bloque local, cada proceso necesita conocer una fila o columna de su vecino. Esto se logra agregando una capa de celdas halo alrededor del bloque real:

$$\text{tamaño real:} \quad \text{local\_nx} \times \text{local\_ny}$$
$$\text{tamaño con halos:} \quad (\text{local\_nx}+2) \times (\text{local\_ny}+2)$$

Los datos reales ocupan los índices $i = 1,\ldots,\text{local\_nx}$ y $j = 1,\ldots,\text{local\_ny}$. Las posiciones $i=0$, $i=\text{local\_nx}+1$, $j=0$ y $j=\text{local\_ny}+1$ son los halos, que se rellenan con datos del proceso vecino correspondiente antes de cada iteración.

En cada iteración, cada proceso intercambia sus bordes con sus cuatro vecinos: arriba, abajo, izquierda y derecha.

+ Filas (bordes superior e inferior): los elementos de una fila son contiguos en memoria, por lo que se pueden enviar directamente con `MPI_Isend` indicando `local_ny` elementos de tipo `MPI_DOUBLE`.

* Columnas(bordes izquierdo y derecho): los elementos de una columna no son contiguos en memoria. En el almacenamiento por filas, dos elementos consecutivos de una misma columna están separados por $\text{local\_ny}+2$ posiciones. Si se intentara enviar una columna con `MPI_DOUBLE`, habría que copiarla manualmente a un buffer auxiliar, lo cual es ineficiente.

La solución es usar un tipo derivado que describe este patrón de memoria no contigua:

```c
MPI_Type_vector(local_nx, 1, local_ny + 2, MPI_DOUBLE, &col_type);
MPI_Type_commit(&col_type);
```

Los parámetros significan:
- `local_nx`: número de bloques (uno por fila del bloque local)
- `1`: tamaño de cada bloque (un solo elemento por fila)
- `local_ny + 2`: separación en elementos entre el inicio de un bloque y el siguiente (*stride*)
- `MPI_DOUBLE`: tipo base

Con este tipo, MPI recorre la memoria saltando de elemento en elemento a lo largo de la columna, sin necesidad de copias intermedias.

Una ventaja del comunicador cartesiano es que `MPI_Cart_shift` devuelve `MPI_PROC_NULL` automáticamente cuando un vecino no existe (cuando el proceso está en el borde de la grilla). Las operaciones `MPI_Isend`/`MPI_Irecv` con destino `MPI_PROC_NULL` se ignoran silenciosamente, lo que simplifica el código.

Las condiciones de borde de Dirichlet ($\phi = 0$ en el borde global) se aplican solo en los procesos que ocupan el borde de la grilla cartesiana, identificados por sus coordenadas:

- $c_x = 0$: borde superior global
- $c_x = d_x - 1$: borde inferior global  
- $c_y = 0$: borde izquierdo global
- $c_y = d_y - 1$: borde derecho global

La estructura general del algoritmo por iteración es:

1. Intercambiar halos con los cuatro vecinos (comunicación no bloqueante)
2. Esperar a que todas las comunicaciones terminen (`MPI_Waitall`)
3. Actualizar `data_new` con la regla de Jacobi usando los halos ya recibidos
4. Imponer condiciones de borde en los procesos del borde global
5. Intercambiar punteros `data` ↔ `data_new`

#### c. Compare the runtime of the algorithm that employs a one-dimensional distribution of tiles and your novel two-dimensional partition for a reasonable number of processes. Can you back up your experimental measurements with theoretical ideas?

**1D** (bloques de filas):

Cada proceso solo tiene vecinos arriba y abajo, y cada mensaje contiene una fila completa de largo $N$:

$$V_{\text{1D}} = 2 \times N \text{ elementos}$$

**2D** (bloques rectangulares):

Con una grilla de $d_x \times d_y$ procesos, cada bloque local tiene tamaño $\frac{N}{d_x} \times \frac{N}{d_y}$. Cada proceso comunica cuatro bordes:

$$V_{\text{2D}} = 2 \times \frac{N}{d_x} + 2 \times \frac{N}{d_y} \text{ elementos}$$

Para una grilla cuadrada $d_x = d_y = \sqrt{P}$:

$$V_{\text{2D}} = \frac{4N}{\sqrt{P}}$$

Comparando con la partición 1D, donde con $P$ procesos cada mensaje tiene $N$ elementos:

$$\frac{V_{\text{2D}}}{V_{\text{1D}}} = \frac{4N/\sqrt{P}}{2N} = \frac{2}{\sqrt{P}}$$

Para $P > 4$, la partición 2D comunica menos datos por proceso que la 1D.

Luego, el tiempo de comunicación de un mensaje de $m$ elementos puede modelarse con la fórmula estándar de latencia más ancho de banda:

$$t_{\text{comm}} = \alpha + \beta \cdot m$$

donde $\alpha$ es la latencia de inicio del mensaje y $\beta$ es el tiempo por elemento transmitido.

**1D:** cada proceso envía 2 mensajes de tamaño $N$:

$$T_{\text{comm}} = 2(\alpha + \beta N)$$

**2D:** cada proceso envía 4 mensajes de tamaño $\frac{N}{\sqrt{P}}$ (grilla cuadrada):

$$T_{\text{comm}} = 4\left(\alpha + \beta \frac{N}{\sqrt{P}}\right)$$

Cuando $P$ es grande, el término $\beta \frac{N}{\sqrt{P}}$ se vuelve pequeño y el costo de comunicación queda dominado por la latencia $\alpha$. En ese régimen la partición 2D paga el doble de latencias, pero el volumen de datos por mensaje cae como $1/\sqrt{P}$, lo que resulta en un costo total menor para $P$ suficientemente grande.

Por ultimo, el costo aritmético por iteración es idéntico en ambas particiones, cada proceso actualiza $\frac{N^2}{P}$ puntos, aplicando 5 sumas/restas y 1 multiplicación por punto:

$$T_{\text{comp}} = \frac{6 N^2}{P} \cdot t_{\text{flop}}$$

Este término escala perfectamente con $P$ en ambos casos, por lo que la diferencia de rendimiento entre las dos particiones proviene solo de la comunicación**

Adicionalmente, se ejecutaron ambas versiones (1D y 2D) con $P = 4$ procesos, $N = 400$ y $1000$ iteraciones. 

La suma $|\text{data}|$ es idéntica en ambos casos, lo que confirma que las dos implementaciones producen el mismo resultado numérico.

Con $P = 4$, la partición 1D resulta más rápida. Esto es consistente con la teoría: con pocos procesos el cómputo domina y la comunicación es pequeña en ambos casos, pero la partición 2D tiene el overhead adicional de gestionar 4 vecinos en lugar de 2, y de usar el tipo derivado `col_type` para las columnas.

El volumen comunicado por proceso en este caso es:
- **1D:** $2 \times 400 = 800$ elementos por iteración
- **2D:** $2 \times 200 + 2 \times 200 = 800$ elementos por iteración

Con $P = 4$ y grilla $2 \times 2$, ambas particiones comunican exactamente el mismo volumen, por lo que la ventaja de la 2D aún no aparece. Esta ventaja se vuelve visible a partir de $P > 4$, cuando $\frac{4N}{\sqrt{P}} < 2N$, es decir, cuando el volumen comunicado por la 2D comienza a ser estrictamente menor.